# rossby1: Rossby waves in a one-layer model on the beta plane
pyOM2: py_config/rossby1.py

One layer with the equivalent depth c_n^2/g, implicit free surface, closed walls
in y. Initial condition is the geostrophically balanced channel mode
eta = eta0 cos(kx) sin(ly) with l = pi/Ly and y measured from the southern wall,
so that v vanishes at both walls.

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('PYOM3_ROOT', os.path.abspath('../..')))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.pe_realistic.model import model

In [ ]:
beta, f0   = 1e-10, 1e-4
cn         = 2.0                    # gravity wave speed
Rn         = cn/f0                  # Rossby radius
nx, ny, nz = 32, 32, 1
dx         = Rn/10
dt         = Rn/cn*0.25
nsteps     = 4000                   # 116 days
plot_every = 100
kx, ly     = 2*np.pi/(nx*dx), np.pi/(ny*dx)   # zonal and meridional wavenumber
eta0       = 0.01                   # initial amplitude of p_s = g eta

par = dict(nx=nx, ny=ny, nz=nz, dxt=dx, dyt=dx, dzt=cn**2/9.81, dt=dt,
           coord_degree=False, set_zero_at_surface=True,
           periodic_in_x=True, periodic_in_y=False,
           eq_of_state=0, implicit_free_surface=True, explicit_free_surface=False,
           epsilon=1e-12, max_iterations=5000, superbee_advection=True)

In [ ]:
class rossby1(model):

    def set_coriolis(self):
        f = f0 + beta*self.yt
        self.coriolis_t = OM.np.asarray(f[:, None] + 0*self.xt[None, :])

    def set_initial_conditions(self):
        # eta on T points, u on (x_u, y_t), v on (x_t, y_u), all in geostrophic balance with f(y).
        # The walls are at yu_gl[0]-dx and yu_gl[-1], so the meridional structure sin(ly*Y)
        # uses Y measured from the southern wall.
        ys = self.yu_gl[0] - dx
        X, Y   = np.meshgrid(self.xt_gl, self.yt_gl - ys)
        p_s    = eta0*np.cos(kx*X)*np.sin(ly*Y)
        Xu, Yt = np.meshgrid(self.xu_gl, self.yt_gl - ys)
        u      = -(ly/(f0+beta*(Yt+ys)))*eta0*np.cos(kx*Xu)*np.cos(ly*Yt)
        Xt, Yv = np.meshgrid(self.xt_gl, self.yu_gl - ys)
        v      = -(kx/(f0+beta*(Yv+ys)))*eta0*np.sin(kx*Xt)*np.sin(ly*Yv)
        zeros  = np.zeros((nz, ny, nx))
        super().set_initial_conditions(u[None], v[None], zeros, zeros, p_s)

In [ ]:
M = rossby1(par)
M.set_coriolis()
M.set_topography()
M.prepare_for_run()
M.set_initial_conditions()
h = M.halo
hov = []                            # p_s along the middle row for the Hovmoeller diagram

In [ ]:
def plot(t, tau):
    clear_output(wait=True)
    p_s = M.p_s[tau, h:-h, h:-h]
    hov.append((t, p_s[ny//2]*1))
    plt.figure(figsize=(5, 4))
    plt.pcolormesh(M.xt_gl/1e3, M.yt_gl/1e3, p_s/9.81*100, cmap='RdBu_r', vmin=-0.1, vmax=0.1)
    plt.colorbar(label='$\\eta$ [cm]'); plt.xlabel('x [km]'); plt.ylabel('y [km]')
    plt.title('t = %.1f days' % (t/86400)); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step_implicit_pressure(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt, tau)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
# Hovmoeller diagram: phase propagation westward, the channel Rossby wave phase speed for comparison
t = np.array([a[0] for a in hov]); p = np.array([a[1] for a in hov])
c_rossby = -beta/(kx**2 + ly**2 + 1/Rn**2)
plt.figure(figsize=(6, 5))
plt.pcolormesh(M.xt_gl/1e3, t/86400, p/9.81*100, cmap='RdBu_r')
plt.plot(((c_rossby*t) % (nx*dx))/1e3, t/86400, 'k.', ms=2)
plt.colorbar(label='$\\eta$ [cm]'); plt.xlabel('x [km]'); plt.ylabel('t [days]')
plt.title('phase speed %.4f m/s' % c_rossby); plt.show()